## 📚 Step 0: Import libraries

Load core libraries for data handling, modelling, and visualisation.

In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

## 📥 Step 1: Load dataset

Import the dataset and standardise column names for consistency.

In [2]:
# Load dataset
df = pd.read_csv('site_leaching_data.csv')

# Ensure datetime is parsed
df["Date"] = pd.to_datetime(df["Date"])

## ⚙️ Step 2: Feature Engineer

In [ ]:
# Define interaction and nonlinear terms
df['Month'] = df['Date'].dt.month
df["CNxDO"] = df["CN_Concentration_PL_01"] * df["Dissolved_Oxygen_PL_01"]
df["Grade_x_Fines_Sqrt"] = np.sqrt(df["Grade_x_Fines"])
df['Gold_Liberation'] = np.clip(df['Grade_x_Fines_Sqrt'], 0, 1)
df['Grade_x_Liberation'] = df['Au_Grade'] * df['Gold_Liberation']
df['Fines_Fraction'] = df['Leach_Feed_Lt_75um_Day'].replace(0, np.nan)
df['Daily_Tonnage'] = df['Leach_Feed_Dry_t']
df['CNxLiberation'] = df['CN_Concentration_PL_01'] * df['Gold_Liberation']
df['DOxGrade'] = df['Dissolved_Oxygen_PL_01'] * df['Au_Grade']

## Create training & testing datasets

In [4]:
# Split the dataset into training and testing sets
df_train = df[(df["Date"] >= "2025-01-01") & (df["Date"] <= "2025-04-30")].copy()
df_test = df[df["Date"] >= "2025-05-01"].copy()

print("Training dataset shape:", df_train.shape)
print(f"From {df_train['Date'].min()} to {df_train['Date'].max()}")
print("Testing dataset shape:", df_test.shape)
print(f"From {df_test['Date'].min()} to {df_test['Date'].max()}")

Training dataset shape: (120, 160)
From 2025-01-01 00:00:00 to 2025-04-30 00:00:00
Testing dataset shape: (55, 160)
From 2025-05-01 00:00:00 to 2025-06-24 00:00:00


## 📊 Step 3: Define plotting helpers

In [5]:
# Create a parity plot
def comparison_scatter_ideal_line(df, x_col, y_col, xlabel, ylabel, title,
                                  fig_size=(6, 6), title_fontsize=16,
                xbounds=None, ybounds=None,
                alpha=0.6, ideal_line_color='red', ideal_line_style='--'):
    """
    Create a scatter plot with an ideal line (y=x) for comparison.
    Parameters:
    - df: DataFrame containing the data.
    - x_col: Column name for x-axis data.
    - y_col: Column name for y-axis data.
    - xlabel: Label for x-axis.
    - ylabel: Label for y-axis.
    - title: Title of the plot.
    - fig_size: Size of the figure.
    - title_fontsize: Font size for the title.
    - xbounds: Optional bounds for x-axis.
    - ybounds: Optional bounds for y-axis.
    - alpha: Transparency level for the scatter points.
    - ideal_line_color: Color of the ideal line.
    - ideal_line_style: Style of the ideal line (e.g., '--', '-.', etc.).
    """

    if not xbounds: xbounds=[df[x_col].min(), df[x_col].max()]
    if not ybounds: ybounds=[df[y_col].min(), df[y_col].max()]
    plt.figure(figsize=fig_size)
    sns.set_theme(style="whitegrid")
    sns.scatterplot(x=x_col, y=y_col, data=df, alpha=alpha)
    plt.plot(xbounds, ybounds,
             color=ideal_line_color, linestyle=ideal_line_style, label='Ideal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title, fontsize=title_fontsize)
    plt.legend()
    plt.grid(False)
    plt.tight_layout()
    plt.show()

## Build Recovery Model

In [9]:
# ---------------------------------------------
# STEP 1: Kinetic Model Fitting
# ---------------------------------------------

RECOVERY_CEILING = 0.95  # or try 0.95 later

def fit_kinetic_parameters(df, manual_params=None):
    if manual_params is not None:
        print(f"📌 Using manual parameters: {manual_params}")
        return manual_params

    def objective(params, df):
        k_base, cn_exp, do_exp = params
        cn = df['CN_Concentration_PL_01'].clip(lower=1)
        do = df['Dissolved_Oxygen_PL_01'].clip(lower=0.1)
        grade = df['Au_Grade']
        liberation = df['Gold_Liberation']
        kinetic_recovery = RECOVERY_CEILING * (1 - np.exp(-k_base * grade * liberation 
                                                          * cn**cn_exp * do**do_exp)) * 100
        error = kinetic_recovery - df['Au_Recovery_pct']
        bias = np.mean(error)
        spread = np.std(error)
        return np.abs(bias) + 0.5 * spread

    bounds = [(1e-5, 0.5), (0.1, 4.0), (0.1, 4.0)]
    initial_guess = [0.005, 1.0, 1.0]
    result = minimize(objective, initial_guess, args=(df,), method='L-BFGS-B', bounds=bounds)

    if result.success:
        k_base, cn_exp, do_exp = result.x
        print(f"✅ Fitted kinetic params: k_base={k_base:.4e}, CN_exp={cn_exp:.3f}, DO_exp={do_exp:.3f}")
        return k_base, cn_exp, do_exp
    else:
        raise RuntimeError("Kinetic parameter fitting failed.")

def apply_kinetic_model(df, k_base, cn_exp, do_exp):
    cn = df['CN_Concentration_PL_01'].clip(lower=1)
    do = df['Dissolved_Oxygen_PL_01'].clip(lower=0.1)
    grade = df['Au_Grade']
    liberation = df['Gold_Liberation']

    # Recovery models
    kinetic_pct = RECOVERY_CEILING * (1 - np.exp(-k_base * grade * liberation 
                                                 * cn**cn_exp * do**do_exp)) * 100
    kinetic_grams = grade * (kinetic_pct / 100)

    df['Kinetic_Recovery_pct'] = kinetic_pct
    df['Kinetic_Au_Recovery'] = kinetic_grams
    return df

# ---------------------------------------------
# STEP 2: Residual Learning
# ---------------------------------------------

def residual_model_training(df_train, df_test, features):
    # Apply centered residual logic
    df_train['Smoothed_Residual'] = df_train['Au_Recovery_pct'] - df_train['Kinetic_Recovery_pct']
    mean_bias = df_train['Smoothed_Residual'].mean()
    df_train['Centered_Residual'] = df_train['Smoothed_Residual'] - mean_bias
    print(f"📉 Mean residual bias: {mean_bias:.2f} percentage points")

    # Train model
    X_train = df_train[features]
    y_train = df_train['Centered_Residual']

    model = GradientBoostingRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.8,
        random_state=42
    )


    model.fit(X_train, y_train)

    # Predict on holdout
    df_test['Kinetic_Recovery_pct'] = apply_kinetic_model(df_test.copy(), 
                                                          k_base, cn_exp, do_exp)['Kinetic_Recovery_pct']
    df_test['Centered_Predicted_Residual'] = model.predict(df_test[features])
    df_test['Corrected_Recovery'] = (
        df_test['Kinetic_Recovery_pct'] +
        df_test['Centered_Predicted_Residual'] +
        mean_bias * 0.5
    )
    df_test['Actual_Recovery'] = df_test['Au_Recovery_pct']

    # Evaluate
    r2 = r2_score(df_test['Actual_Recovery'], df_test['Corrected_Recovery'])
    mae = mean_absolute_error(df_test['Actual_Recovery'], df_test['Corrected_Recovery'])
    rmse = root_mean_squared_error(df_test['Actual_Recovery'], df_test['Corrected_Recovery'])
    print(f"\n📈 Residual Model Performance:\n  R²: {r2:.3f}\n  "
          f"MAE: {mae:.3f} % recovery\n  RMSE: {rmse:.3f}")

    return model, df_test

# ---------------------------------------------
# RUN PIPELINE
# ---------------------------------------------

# Step 1: Fit kinetics on training data only
# manual_params = (5.7833e-03, -7.950, 1.825)  # k_base, cn_exp, do_exp
# k_base, cn_exp, do_exp = fit_kinetic_parameters(df_train, manual_params=manual_params)

k_base, cn_exp, do_exp = fit_kinetic_parameters(df_train)

# Step 2: Apply same kinetic model to both datasets
df_train = apply_kinetic_model(df_train, k_base, cn_exp, do_exp)
df_test = apply_kinetic_model(df_test, k_base, cn_exp, do_exp)

print(f"🔍 Max predicted kinetic recovery: {df_test['Kinetic_Recovery_pct'].max():.2f}%")


# Step 3: Compute actual - kinetic residuals
df_train['Recovery_Residual'] = df_train['Au_Recovery_pct'] - df_train['Kinetic_Au_Recovery']
df_test['Recovery_Residual'] = df_test['Au_Recovery_pct'] - df_test['Kinetic_Au_Recovery']

# Step 4: Smooth residuals
df_train['Smoothed_Residual'] = df_train['Recovery_Residual'].rolling(window=7, min_periods=1).mean()


# Step 5: Residual model training
features = [
    'Leach_Feed_Throughput_M3Hr',
    'Percentsolids',
    'Grade_x_Fines_Sqrt',
    'Dissolved_Oxygen_PL_01',
    'CN_Concentration_PL_01',
    'Particle_Size',
    'Au_Grade',
    'Grade_x_Liberation',
    'Daily_Tonnage',
    'Fines_Fraction',
    'CNxLiberation',
    'DOxGrade'
]


residual_model, results_df = residual_model_training(df_train, df_test.copy(), features)

# Feature importance
importance = residual_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importance
}).sort_values(by='Importance', ascending=False)

# print("\n📊 Feature Importance:"
#       "\n", feature_importance_df)

# Diagnostic: Bias vs Grade
# df_test['Prediction_Bias'] = df_test['Corrected_Recovery'] - df_test['Au_Recovery_pct']

# sns.scatterplot(data=df_test, x='Au_Grade', y='Prediction_Bias')
# plt.axhline(0, linestyle='--', color='red')
# plt.title("Prediction Bias vs Au Grade")



# # Step 6: Visualise

# comparison_scatter_ideal_line(
#     results_df, 'Actual_Recovery', 'Corrected_Recovery',
#     xlabel="Actual Recovery (%)",
#     ylabel="Corrected Recovery (%)",
#     title="Corrected vs Actual Recovery",
#     fig_size=(6, 6)
# )

✅ Fitted kinetic params: k_base=5.0000e-01, CN_exp=1.000, DO_exp=2.259
🔍 Max predicted kinetic recovery: 95.00%
📉 Mean residual bias: -6.65 percentage points

📈 Residual Model Performance:
  R²: -4.173
  MAE: 3.891 % recovery
  RMSE: 4.165


In [ ]:
from sklearn.model_selection import GridSearchCV

# Center the residuals in the training set
mean_bias = df_train['Recovery_Residual'].mean()
df_train['Centered_Residual'] = df_train['Recovery_Residual'] - mean_bias

# Features used for residual learning
features = [
    'Leach_Feed_Throughput_M3Hr',
    'Percentsolids',
    'Grade_x_Fines_Sqrt',
    'Dissolved_Oxygen_PL_01',
    'CNxDO',
    'Particle_Size'
]

# Train residual model
X_train = df_train[features]
y_train = df_train['Centered_Residual']
residual_model = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
residual_model.fit(X_train, y_train)

# Apply to test set
X_test = df_test[features]
df_test['Centered_Predicted_Residual'] = residual_model.predict(X_test)
df_test['Corrected_Recovery'] = df_test['Kinetic_Recovery_pct'] + df_test['Centered_Predicted_Residual'] + mean_bias

# Evaluate performance
actual = df_test['Au_Recovery_pct']
predicted = df_test['Corrected_Recovery']
r2 = r2_score(actual, predicted)
mae = mean_absolute_error(actual, predicted)
rmse = root_mean_squared_error(actual, predicted)

# Print performance metrics
print(f"\n📊 Residual Model Performance on Test Set:\n  R²: {r2:.3f}\n  "
      f"MAE: {mae:.3f} % recovery\n  RMSE: {rmse:.3f}")

# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3, 4],
    'learning_rate': [0.05, 0.1, 0.2]
}

# Use 5-fold CV to evaluate combinations
gbr = GradientBoostingRegressor(random_state=42)
grid_search = GridSearchCV(gbr, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_

# Re-apply best model to test set
df_test['Centered_Predicted_Residual'] = best_model.predict(X_test)
df_test['Corrected_Recovery'] = df_test['Kinetic_Recovery_pct'] + df_test['Centered_Predicted_Residual'] + mean_bias

# Evaluate new performance
r2_best = r2_score(actual, df_test['Corrected_Recovery'])
mae_best = mean_absolute_error(actual, df_test['Corrected_Recovery'])
rmse_best = root_mean_squared_error(actual, df_test['Corrected_Recovery'])

print(f"\n🏆 Best Model Performance:\n  R²: {r2_best:.3f}\n  "
        f"MAE: {mae_best:.3f} % recovery\n  RMSE: {rmse_best:.3f}")


In [ ]:
features = [
    'Au_Grade',
    'Leach_Feed_Throughput_M3Day',
    'Percentsolids',
    'Dissolved_Oxygen_Profile_Ppm_Leach_Tank_01',
    'Ph_Leach_Tank_01',
    'CNxDO'
]


In [ ]:
# Combine datasets (remove duplicates if any)
df_all = pd.concat([df_train, df_test]).drop_duplicates(subset="Date").reset_index(drop=True)

# Check for key columns required for CN models
required_columns = [
    'Leach_Feed_Dry_t', 'Leach_Feed_Throughput_M3Day', 'Au_Grade',
    'Dissolved_Oxygen_PL_01',
    'Ph_Leach_Tank_01',
    'CN_Concentration_PL_01',
    'CN_Tailings_Concentration',
    'Percentsolids',
    'Dissolved_Au_Metal_Profile_G_Leach_Tank_01'
]

missing_columns = [col for col in required_columns if col not in df_all.columns]

# If no missing columns, calculate target variables
if not missing_columns:
    df_all["CN_Consumption_kg_t"] = (df_all["NaCN_Used_Kg"] * 1000) / df_all["Leach_Feed_Dry_t"]
    df_all["CN_Loss_ppm"] = df_all["CN_Concentration_PL_01"] - df_all["CN_Tailings_Concentration"]

    # CN × DO interaction term
    df_all["CNxDO"] = (
        df_all["CN_Concentration_PL_01"] *
        df_all["Dissolved_Oxygen_PL_01"]
    )

    # Filter rows where we have valid CN target values
    df_clean = df_all.dropna(subset=["CN_Consumption_kg_t", "CN_Loss_ppm"])
    
    df_clean
else:
    df_clean = None

missing_columns, df_clean.shape if df_clean is not None else "Data not ready"


## Build Cyanide Consumption Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import numpy as np

# Define features and target for CN Consumption model
features = [
    'Au_Grade',
    'Leach_Feed_Throughput_M3Day',
    'Percentsolids',
    'Dissolved_Oxygen_Profile_Ppm_Leach_Tank_01',
    'Ph_Leach_Tank_01',
    'CNxDO'
]
target = 'CN_Consumption_kg_t'

# Drop rows with missing values in selected features
df_model = df_clean.dropna(subset=features + [target])

# Split into train/test sets
X = df_model[features]
y = df_model[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Gradient Boosting model
model = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2, mae, rmse

# Display performance metrics
print(f"R²: {r2:.3f}, MAE: {mae:.3f} kg/t, RMSE: {rmse:.3f} kg/t")

import matplotlib.pyplot as plt
import seaborn as sns

# Create parity plot for CN Consumption prediction
plt.figure(figsize=(6, 6))
sns.set_theme(style="whitegrid")
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         color='red', linestyle='--', label='Ideal')
plt.xlabel("Actual CN Consumption (kg/t)")
plt.ylabel("Predicted CN Consumption (kg/t)")
plt.title("Actual vs Predicted Cyanide Consumption")
plt.legend()
# Add Performance Metrics
plt.text(0.05, 0.95, f"R²: {r2:.3f}\nMAE: {mae:.3f} kg/t\nRMSE: {rmse:.3f} kg/t",
         transform=plt.gca().transAxes, fontsize=12,
            verticalalignment='top', bbox=dict(facecolor='white', alpha=0.8, edgecolor='black'))
plt.tight_layout()
plt.show()


In [ ]:
importances = model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by="Importance", ascending=False)

# Plot feature importance
plt.figure(figsize=(8, 5))
sns.barplot(data=feature_importance_df, x="Importance", y="Feature", hue="Feature", palette="viridis")
plt.title("Feature Importance – CN Consumption Model")
plt.xlabel("Relative Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

feature_importance_df.reset_index(drop=True)


## CN Loss Model

In [ ]:
# Define features and target for CN Loss model
loss_features = [
    'Au_Grade',
    'Dissolved_Au_Metal_Profile_G_Leach_Tank_01',
    'Dissolved_Oxygen_PL_01',
    'Percentsolids',
    'Ph_Leach_Tank_01',
    'CN_Concentration_PL_01',
    'Leach_Feed_Throughput_M3Day'
]
loss_target = 'CN_Loss_ppm'

# Drop rows with missing values in selected features and target
df_loss_model = df_clean.dropna(subset=loss_features + [loss_target])

# Split into train/test sets
X_loss = df_loss_model[loss_features]
y_loss = df_loss_model[loss_target]
X_loss_train, X_loss_test, y_loss_train, y_loss_test = train_test_split(
    X_loss, y_loss, test_size=0.2, random_state=42
)

# Train Gradient Boosting model
loss_model = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
loss_model.fit(X_loss_train, y_loss_train)

# Predict and evaluate
y_loss_pred = loss_model.predict(X_loss_test)
r2_loss = r2_score(y_loss_test, y_loss_pred)
mae_loss = mean_absolute_error(y_loss_test, y_loss_pred)
rmse_loss = np.sqrt(mean_squared_error(y_loss_test, y_loss_pred))

# Display performance metrics
print(f"R²: {r2_loss:.3f}, MAE: {mae_loss:.3f} ppm, RMSE: {rmse_loss:.3f} ppm")

In [ ]:
from scipy.optimize import curve_fit
import numpy as np

# Define the nonlinear function to fit (based on CN and DO)
def cyanide_loss_model(X, a, b, c, d):
    CN_input, DO = X
    return a * (CN_input ** b) * (DO ** c) + d

# Extract and clean relevant columns
df_curve = df_clean.dropna(subset=["CN_Loss_ppm", "CN_Concentration_PL_01", "Dissolved_Oxygen_PL_01"])

# Input features and target
X_data = np.array([
    df_curve["CN_Concentration_PL_01"],
    df_curve["Dissolved_Oxygen_PL_01"]
])
y_data = df_curve["CN_Loss_ppm"].values

# Fit the nonlinear curve
initial_guess = [1, 1, 1, 0]  # Starting values for a, b, c, d
params, _ = curve_fit(cyanide_loss_model, X_data, y_data, p0=initial_guess, maxfev=10000)

# Predict using the fitted curve
y_fit = cyanide_loss_model(X_data, *params)

# Calculate fit metrics
r2_fit = r2_score(y_data, y_fit)
mae_fit = mean_absolute_error(y_data, y_fit)
rmse_fit = np.sqrt(mean_squared_error(y_data, y_fit))

params, r2_fit, mae_fit, rmse_fit
# Display performance metrics
print(f"Fitted parameters: {params}")
print(f"R²: {r2_fit:.3f}, MAE: {mae_fit:.3f} ppm, RMSE: {rmse_fit:.3f} ppm")

The nonlinear curve fitting yielded the following model:

$$CN\_Loss=1.585 ⋅ (CN_input)1.13⋅(DO)−0.60−119.47$$

#### 📊 Fit Metrics:
R²: 0.493 — comparable to the ML model

MAE: 96.36 ppm

RMSE: 130.72 ppm

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 1. Parity Plot
plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_data, y=y_fit, alpha=0.6)
plt.plot([y_data.min(), y_data.max()], [y_data.min(), y_data.max()],
         color='red', linestyle='--', label='Ideal')
plt.xlabel("Actual CN Loss (ppm)")
plt.ylabel("Predicted CN Loss (ppm)")
plt.title("Actual vs Predicted Cyanide Loss (Curve Fit)")
plt.legend()
plt.tight_layout()
plt.show()

# 2. 3D Surface Plot
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
cn_range = np.linspace(df_curve["CN_Concentration_PL_01"].min(),
                       df_curve["CN_Concentration_PL_01"].max(), 50)
do_range = np.linspace(df_curve["Dissolved_Oxygen_PL_01"].min(),
                       df_curve["Dissolved_Oxygen_PL_01"].max(), 50)
cn_grid, do_grid = np.meshgrid(cn_range, do_range)
z_grid = cyanide_loss_model((cn_grid, do_grid), *params)

ax.plot_surface(cn_grid, do_grid, z_grid, cmap="viridis", alpha=0.9)
ax.set_xlabel("CN Input (ppm)")
ax.set_ylabel("DO (ppm)")
ax.set_zlabel("Predicted CN Loss (ppm)")
ax.set_title("3D Surface: CN Loss vs CN Input & DO")
plt.tight_layout()
plt.show()

# 3. 2D Slice: Fix DO, vary CN
fixed_do = df_curve["Dissolved_Oxygen_PL_01"].median()
cn_values = np.linspace(df_curve["CN_Concentration_PL_01"].min(),
                        df_curve["CN_Concentration_PL_01"].max(), 100)
loss_slice_cn = cyanide_loss_model((cn_values, np.full_like(cn_values, fixed_do)), *params)

plt.figure(figsize=(7, 4))
plt.plot(cn_values, loss_slice_cn)
plt.xlabel("CN Input (ppm)")
plt.ylabel("Predicted CN Loss (ppm)")
plt.title(f"Predicted CN Loss vs CN Input (DO fixed at {fixed_do:.2f} ppm)")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Define extended nonlinear function
def extended_cyanide_loss_model(X, a, b, c, d, e, f):
    CN_input, DO, Grade, PercSolids = X
    return a * (CN_input ** b) * (DO ** c) * (Grade ** d) * (PercSolids ** e) + f

# Drop rows with needed fields
df_ext = df_clean.dropna(subset=[
    "CN_Loss_ppm",
    "CN_Concentration_PL_01",
    "Dissolved_Oxygen_PL_01",
    "Au_Grade",
    "Percentsolids"
])

# Prepare input arrays
X_ext = np.array([
    df_ext["CN_Concentration_PL_01"],
    df_ext["Dissolved_Oxygen_PL_01"],
    df_ext["Au_Grade"],
    df_ext["Percentsolids"]
])
y_ext = df_ext["CN_Loss_ppm"].values

# Initial guess for parameters: a, b, c, d, e, f
initial_guess_ext = [1, 1, 1, 1, 1, 0]

# Fit the curve
params_ext, _ = curve_fit(extended_cyanide_loss_model, X_ext,
                          y_ext, p0=initial_guess_ext, maxfev=20000)

# Predict with extended model
y_ext_fit = extended_cyanide_loss_model(X_ext, *params_ext)

# Compute fit metrics
r2_ext = r2_score(y_ext, y_ext_fit)
mae_ext = mean_absolute_error(y_ext, y_ext_fit)
rmse_ext = np.sqrt(mean_squared_error(y_ext, y_ext_fit))

# Display performance metrics
print(f"Intersect: {params_ext[0]:.3f}")
print(f"CN exponent: {params_ext[1]:.3f}")
print(f"DO exponent: {params_ext[2]:.3f}")
print(f"Grade exponent: {params_ext[3]:.3f}")
print(f"Percentsolids exponent: {params_ext[4]:.3f}")
print(f"Fitted constant: {params_ext[5]:.3f}")
print(f"\nR²: {r2_ext:.3f}, MAE: {mae_ext:.3f} ppm, RMSE: {rmse_ext:.3f} ppm")


Fitted Equation:
$$CN_{Loss} = 0.610 ⋅ CN^{1.084} ⋅ DO^{−0.258} ⋅ Grade^{0.725} ⋅ Solids^{− 0.065} −129.60$$

## Visalise Key Features

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_fit = df_ext.dropna(subset=[
    "CN_Loss_ppm", "CN_Concentration_PL_01", "Dissolved_Oxygen_PL_01",
    "Au_Grade", "Percentsolids"
])
X_ext = np.array([
    df_fit["CN_Concentration_PL_01"],
    df_fit["Dissolved_Oxygen_PL_01"],
    df_fit["Au_Grade"],
    df_fit["Percentsolids"]
])
y_ext = df_fit["CN_Loss_ppm"].values

params_ext, _ = curve_fit(extended_cyanide_loss_model, X_ext, y_ext, p0=[1, 1, 1, 1, 1, 0], maxfev=20000)

# Predict and residuals
df_fit["CN_Loss_Predicted"] = extended_cyanide_loss_model(X_ext, *params_ext)
df_fit["CN_Loss_Residual"] = df_fit["CN_Loss_ppm"] - df_fit["CN_Loss_Predicted"]
df_fit["Smoothed_CN_Loss_Residual"] = df_fit["CN_Loss_Residual"].rolling(window=7, min_periods=1).mean()

# Select residual and key features for plotting
residual_data = df_fit.copy()
residual_data["Residual"] = residual_data["CN_Loss_ppm"] - residual_data["CN_Loss_Predicted"]
features_to_plot = [
    "CN_Concentration_PL_01",
    "Dissolved_Oxygen_PL_01",
    "Au_Grade",
    "Percentsolids",
    "Ph_Leach_Tank_01",
    "CN_Consumption_kg_t"
]

# Create scatter plots of residuals vs key features
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
axes = axes.flatten()

for ax, feature in zip(axes, features_to_plot):
    sns.scatterplot(data=residual_data, x=feature, y="Residual", alpha=0.6, ax=ax)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_title(f"Residual vs {feature}")
    ax.set_xlabel(feature)
    ax.set_ylabel("Residual (ppm)")

plt.tight_layout()
plt.show()


#### 🔍 What the plots shows:
* **CN Concentration:** Slight upward curvature in residuals at high CN → potential underfitting at extreme values.
* **DO:** Possible nonlinearity or interaction not captured well (residuals rise again at higher DO).
* **Au Grade:** Some structure — residuals tend to drop with higher grades, suggesting underfitting.
* **Percent Solids:** Appears mostly random.
* **pH and CN Consumption:** Potential weak trends, especially at low pH.

## Log-log transformation
* Linearise power-law or exponential behaviour.
* Apply log() to CN_Loss, CN, DO, Grade, Solids (adding small offset to avoid log(0))

In [ ]:
from sklearn.linear_model import LinearRegression

# Create log-transformed variables (+1 to avoid log(0))
df_log = df_fit.copy()
df_log["log_CN_Loss"] = np.log(df_log["CN_Loss_ppm"] + 1)
df_log["log_CN"] = np.log(df_log["CN_Concentration_PL_01"] + 1)
df_log["log_DO"] = np.log(df_log["Dissolved_Oxygen_PL_01"] + 1)
df_log["log_Grade"] = np.log(df_log["Au_Grade"] + 1)
df_log["log_Solids"] = np.log(df_log["Percentsolids"] + 1)

# Drop any rows with NA values
df_log_clean = df_log.dropna(subset=["log_CN_Loss", "log_CN", "log_DO", "log_Grade", "log_Solids"])

# Prepare data
X_log = df_log_clean[["log_CN", "log_DO", "log_Grade", "log_Solids"]]
y_log = df_log_clean["log_CN_Loss"]

# Fit log-log linear model
log_model = LinearRegression()
log_model.fit(X_log, y_log)

# Predict and evaluate
y_log_pred = log_model.predict(X_log)
r2_log = r2_score(y_log, y_log_pred)
mae_log = mean_absolute_error(y_log, y_log_pred)
rmse_log = np.sqrt(mean_squared_error(y_log, y_log_pred))

# Display performance metrics
print(f"Log-Log Model Performance:\n"
        f"  R²: {r2_log:.3f}\n"
        f"  MAE: {mae_log:.3f} ppm\n"
        f"  RMSE: {rmse_log:.3f} ppm")

## Try XGBoost

In [ ]:
from xgboost import XGBRegressor

# Define input features and target for XGBoost
xgb_features = [
    "CN_Concentration_PL_01",
    "Dissolved_Oxygen_PL_01",
    "Au_Grade",
    "Percentsolids"
]
xgb_target = "CN_Loss_ppm"

# Prepare cleaned dataset
df_xgb = df_fit.dropna(subset=xgb_features + [xgb_target])
X_xgb = df_xgb[xgb_features]
y_xgb = df_xgb[xgb_target]

# Split into train/test
X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(X_xgb, y_xgb,
                                                                    test_size=0.2, random_state=42)

# Train XGBoost model
xgb_model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1,
                         objective='reg:squarederror', random_state=42)
xgb_model.fit(X_train_xgb, y_train_xgb)

# Predict and evaluate
y_pred_xgb = xgb_model.predict(X_test_xgb)
r2_xgb = r2_score(y_test_xgb, y_pred_xgb)
mae_xgb = mean_absolute_error(y_test_xgb, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test_xgb, y_pred_xgb))

r2_xgb, mae_xgb, rmse_xgb
# Display performance metrics
print(f"XGBoost Model Performance:\n"
        f"  R²: {r2_xgb:.3f}\n"
        f"  MAE: {mae_xgb:.3f} ppm\n"
        f"  RMSE: {rmse_xgb:.3f} ppm")

### Add Engineered Features

In [ ]:
df_xgb["CNxDO"] = df_xgb["CN_Concentration_PL_01"] * df_xgb["Dissolved_Oxygen_PL_01"]
df_xgb["CNxGrade"] = df_xgb["CN_Concentration_PL_01"] * df_xgb["Au_Grade"]
df_xgb["DOxGrade"] = df_xgb["Dissolved_Oxygen_PL_01"] * df_xgb["Au_Grade"]
df_xgb["GradexSolids"] = df_xgb["Au_Grade"] * df_xgb["Percentsolids"]
df_xgb["CNxpH"] = df_xgb["CN_Concentration_PL_01"] * df_xgb["Ph_Leach_Tank_01"]

### Define a New Feature List

In [ ]:
xgb_features_extended = [
    "CN_Concentration_PL_01",
    "Dissolved_Oxygen_PL_01",
    "Au_Grade",
    "Percentsolids",
    "Ph_Leach_Tank_01",
    "CNxDO",
    "CNxGrade",
    "DOxGrade",
    "GradexSolids",
    "CNxpH"
]


### Refit XGB

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Drop missing values
df_xgb_clean = df_xgb.dropna(subset=xgb_features_extended + ["CN_Loss_ppm"])
X = df_xgb_clean[xgb_features_extended]
y = df_xgb_clean["CN_Loss_ppm"]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
xgb_model = XGBRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    objective='reg:squarederror', random_state=42
)
xgb_model.fit(X_train, y_train)

# Predict
y_pred = xgb_model.predict(X_test)

# Evaluate
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R²: {r2:.3f}")
print(f"MAE: {mae:.3f} ppm")
print(f"RMSE: {rmse:.3f} ppm")


### Visualise Actual vs Predicted

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--', label='Ideal')
plt.xlabel("Actual CN Loss (ppm)")
plt.ylabel("Predicted CN Loss (ppm)")
plt.title("XGBoost CN Loss: Actual vs Predicted")
# Add Performance Metrics
plt.text(0.05, 0.95, f"R²: {r2:.3f}\nMAE: {mae:.3f} ppm\nRMSE: {rmse:.3f} ppm",
         transform=plt.gca().transAxes, fontsize=12,
            verticalalignment='top', bbox=dict(facecolor='white', alpha=0.8, edgecolor='black'))
plt.grid(False)
plt.legend()
plt.tight_layout()
plt.show()


### Check Feature Importance

In [ ]:
importances = xgb_model.feature_importances_
feature_importance_df = pd.DataFrame({
    "Feature": xgb_features_extended,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print(feature_importance_df)


### Save the Model

In [ ]:
import joblib

# Save model to file
joblib.dump(xgb_model, "xgb_cn_loss_model.pkl")
print("✅ Model saved to xgb_cn_loss_model.pkl")


### Plot Time Series: Predicted vs Actual CN Loss

In [ ]:
# Reattach predictions to full dataset
df_xgb_clean.loc[y_test.index, "Predicted_CN_Loss"] = y_pred
df_xgb_clean.loc[y_test.index, "Actual_CN_Loss"] = y_test

# Sort by date
df_plot = df_xgb_clean.loc[y_test.index][["Date", "Actual_CN_Loss", "Predicted_CN_Loss"]].sort_values("Date")

# Plot
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.plot(df_plot["Date"], df_plot["Actual_CN_Loss"], label="Actual", marker='o', alpha=0.6)
plt.plot(df_plot["Date"], df_plot["Predicted_CN_Loss"], label="Predicted", marker='x', alpha=0.6)
plt.title("CN Loss: Actual vs Predicted Over Time")
plt.xlabel("Date")
plt.ylabel("CN Loss (ppm)")
plt.legend()
plt.grid(False)
plt.tight_layout()
plt.show()


### SHAP Analysis for Explainability

In [ ]:
import shap

# Re-train on full data to enable SHAP analysis
xgb_model_full = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1,
                              objective='reg:squarederror', random_state=42)
xgb_model_full.fit(X, y)

# SHAP explainer
explainer = shap.Explainer(xgb_model_full)
shap_values = explainer(X)

# Summary plot
shap.summary_plot(shap_values, X, plot_type="bar")


### View per-row SHAP impact (why CN spiked on a certain day)

In [ ]:
shap.plots.waterfall(shap_values[0])  # replace i with index of interest

#### Interpreting the SHAP Waterfall Plot (Per-row)
Shows how the model made a prediction for a single row (e.g. one day of leaching data).

#### 📊 What each part means:

| Element                  | Meaning                                                                         |
| ------------------------ | ------------------------------------------------------------------------------- |
| **Base value** `E[f(X)]` | The average model prediction across the training data (baseline CN loss in ppm) |
| **Each bar**             | Contribution of a specific feature to **push the prediction up or down**        |
| **Blue bars (−)**        | Features that **pull the prediction down** relative to the base value           |
| **Red bars (+)**         | Features that **push the prediction up** relative to the base value             |
| **Feature = value**      | The feature value for that row (e.g. `10317.8 = CNxDO`)                         |
| **Bar length**           | The **magnitude of impact** on the prediction (in ppm)                          |
| **Rightmost value**      | Final model prediction for that row = base value + sum of SHAP contributions    |

#### Example:
If:
* $E[f(X)]$ = 160.5 ppm
* `CNxDO` contributed +46.6 ppm
* `CNxGrade` contributed −30.1 ppm

Then the predicted CN loss is:
$$Predicted CN Loss = 160.5 + 46.6 - 30.1 + … = ~195 ppm$$

This helps understand why a specific prediction was high or low — ideal for explainability to ops/met teams.

### Predict Absolute CN Consumption (kg/day)

In [ ]:
# Drop rows missing NaCN usage
df_cnkg = df_xgb.dropna(subset=["NaCN_Used_Kg"] + xgb_features_extended)

X_kg = df_cnkg[xgb_features_extended]
y_kg = df_cnkg["NaCN_Used_Kg"]

# Train/test split
X_train_kg, X_test_kg, y_train_kg, y_test_kg = train_test_split(X_kg, y_kg, test_size=0.2, random_state=42)

# Train model
model_kg = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1, objective='reg:squarederror', random_state=42)
model_kg.fit(X_train_kg, y_train_kg)

# Predict and evaluate
y_pred_kg = model_kg.predict(X_test_kg)

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
print("NaCN Consumption Model")
print(f"R²: {r2_score(y_test_kg, y_pred_kg):.3f}")
print(f"MAE: {mean_absolute_error(y_test_kg, y_pred_kg):.2f} kg")
print(f"RMSE: {root_mean_squared_error(y_test_kg, y_pred_kg):.2f} kg")
